# Deliverable 3 — The Trump Moment

The President's own disclosed trades (Quiver's Trump Stock Trades endpoint — short history, and that's fine), run through **the same v1 and v2 engines** as Congress. Plus the April 9, 2025 case study: the **"THIS IS A GREAT TIME TO BUY!!!"** Truth Social post (9:37 AM ET) and the tariff-pause announcement (1:18 PM ET), on SPY minute bars from Massive.

Outputs → `output/3-trump-moment/`: `trump_trades_v1_v2.mp4`, `spy_intraday_apr9.mp4`, `trump_trade_results.csv`, `summary_stats.json`

> **Editor note (b-roll, not produced here):** the Trump-accuses-Pelosi-of-insider-trading clip and Truth Social screenshots are external b-roll — human editor sources these.

In [1]:
import sys
import json
import warnings
from pathlib import Path
from datetime import timedelta
from dotenv import load_dotenv

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.animation as mpl_animation
import matplotlib.dates as mdates

REPO_ROOT = Path.cwd().resolve().parent
load_dotenv(REPO_ROOT / '.env')
sys.path.insert(0, str(REPO_ROOT))

from lib import brand, animation, data_quiver, data_massive, backtest
from lib.brand import BG, GREEN, OFF_WHITE, RED, OLIVE, GRID, VCR
from audio_engine import Cue, render_track, ticks_every

warnings.filterwarnings('ignore')
brand.apply_theme()

HOLD_DAYS = 60
PRICE_END = '2026-07-02'
OUT_DIR = Path('../output/3-trump-moment')
OUT_DIR.mkdir(parents=True, exist_ok=True)
AUDIO_DIR = Path('cache/audio')
AUDIO_DIR.mkdir(parents=True, exist_ok=True)
print(f'Output directory: {OUT_DIR.resolve()}')

Output directory: G:\congress-trades-investigation\output\3-trump-moment


In [2]:
# ── Trump's disclosed trades (Quiver) ─────────────────────────────────────────
trump_raw = data_quiver.fetch_trump_trades()
print(f'{len(trump_raw)} disclosed Trump trades '
      f'({trump_raw["TransactionDate"].min().date()} → {trump_raw["TransactionDate"].max().date()})')
print(trump_raw['Transaction'].value_counts().to_string())

# Same cleaning rules as the Congress engines: buys, valid ticker + filing date.
trump_buys = trump_raw[
    trump_raw['Transaction'].str.lower().str.contains('purchase|buy', na=False)
].dropna(subset=['Ticker', 'ReportDate']).copy()
trump_buys['Ticker'] = trump_buys['Ticker'].str.upper().str.strip()
trump_buys = trump_buys[trump_buys['Ticker'].str.match(r'^[A-Z]{1,5}$', na=False)]
trump_buys = trump_buys[trump_buys['Ticker'] != 'DJT']
trump_buys = trump_buys.drop_duplicates(subset=['Ticker', 'TransactionDate']).reset_index(drop=True)
print(f'\n{len(trump_buys)} clean buys | {trump_buys["Ticker"].nunique()} tickers')
trump_buys.head(3)

112 disclosed Trump trades (2026-01-06 → 2026-03-27)
Transaction
Purchase    111
Sale          1

65 clean buys | 53 tickers


,Ticker,Company,Transaction,Amount,ReportDate,TransactionDate,ExcessReturn,Representative
0,AVB,AVALONDAY CMNTYS INC COM,Purchase,"$250,001 - $500,000",2026-05-13,2026-01-06,-2.094519,Donald Trump
1,ARES,ARES MANAGEMENT CORPORATION CL A COM,Purchase,"$250,001 - $500,000",2026-05-13,2026-01-06,-41.236272,Donald Trump
2,ITW,ILLINOIS TOOL WORKS,Purchase,"$500,001 - $1,000,000",2026-05-13,2026-01-06,-0.334279,Donald Trump


In [3]:
# Prices from the shared Massive cache (Congress notebooks already fetched most)
fetch_start = (trump_buys['TransactionDate'].min() - timedelta(days=5)).strftime('%Y-%m-%d')
tickers = ['SPY'] + sorted(trump_buys['Ticker'].unique().tolist())
price_cache = data_massive.load_price_cache(tickers, '2012-06-01', PRICE_END)
spy_prices = price_cache['SPY']

# ── Same engines, presidential subset ─────────────────────────────────────────
# Many filings are recent, so v2 holds often aren't complete yet — those are
# marked to market and flagged Open rather than hidden (extreme honesty).
print('v1 — perfect info (his transaction date):')
tv1 = backtest.run_backtest(trump_buys, price_cache, spy_prices,
                            hold_days=HOLD_DAYS, entry='transaction', allow_open=True)
print(f"  open (hold incomplete): {tv1['Open'].mean():.0%}")
print('\nv2 — reality (filing date + 1):')
tv2 = backtest.run_backtest(trump_buys, price_cache, spy_prices,
                            hold_days=HOLD_DAYS, entry='disclosure', allow_open=True)
print(f"  open (hold incomplete): {tv2['Open'].mean():.0%}")

Price cache: 3778 tickers cached | 0 to fetch
Done: 54 tickers with data | 0 empty (delisted/bad symbol)
v1 — perfect info (his transaction date):
65 trades complete | 0 skipped
Avg return: 0.18% | SPY: 2.23% | Excess: -2.06%
Win rate: 41.54% | Beat SPY: 36.92%
  open (hold incomplete): 0%

v2 — reality (filing date + 1):
65 trades complete | 0 skipped
Avg return: 2.85% | SPY: -0.45% | Excess: 3.30%
Win rate: 53.85% | Beat SPY: 55.38%
  open (hold incomplete): 100%


In [4]:
# ── Visual 1: the "viral" Trump trades under both engines ─────────────────────
tkey = ['Ticker', 'TransactionDate']
tmerged = tv1.merge(tv2, on=tkey, suffixes=('_v1', '_v2'))
tviral = tmerged.nlargest(8, 'TradeReturn_v1').iloc[::-1]

GROW_S, GAP_S, HOLD_S = 1.5, 1.5, 3.0
wav = render_track(
    [Cue('bar_grow', 0.1), Cue('data_blip', GROW_S, gain_db=-4),
     Cue('bar_grow', GROW_S + GAP_S, gain_db=-3),
     Cue('error_buzz', GROW_S + GAP_S + GROW_S, gain_db=-6)],
    duration_s=GROW_S + GAP_S + GROW_S + HOLD_S,
    out_path=AUDIO_DIR / 'd3_viral.wav',
)
animation.animate_paired_bars(
    labels=[f"{r['Ticker']}  {pd.Timestamp(r['TransactionDate']):%b %d %y}"
            for _, r in tviral.iterrows()],
    values_a=tviral['TradeReturn_v1'].tolist(),
    values_b=tviral['TradeReturn_v2'].tolist(),
    title="THE PRESIDENT'S TOP TRADES - HIS ENTRY vs YOURS",
    output_path=OUT_DIR / 'trump_trades_v1_v2.mp4',
    label_a='his transaction day', label_b='when you could act',
    grow_seconds=GROW_S, gap_seconds=GAP_S, hold_seconds=HOLD_S,
    audio_wav=wav,
)

tv1.assign(Engine='v1').to_csv(OUT_DIR / 'trump_trade_results.csv', index=False)
tv2.assign(Engine='v2').to_csv(OUT_DIR / 'trump_trade_results_v2.csv', index=False)

findfont: Failed to find font weight bold, now using 400.


Rendering trump_trades_v1_v2.mp4  (450 frames @ 60 fps)...


Saved trump_trades_v1_v2.mp4  (0.2 MB)


In [5]:
# ── April 9, 2025 — SPY minute bars + the two Truth Social timestamps ─────────
# 9:37 AM ET  "THIS IS A GREAT TIME TO BUY!!! DJT"   (post time per PBS/NPR reporting)
# 1:18 PM ET  90-day tariff pause announced           (post time per PBS/NPR reporting)
APR9 = '2025-04-09'
POST_TIME  = pd.Timestamp(f'{APR9} 09:37', tz='US/Eastern')
PAUSE_TIME = pd.Timestamp(f'{APR9} 13:18', tz='US/Eastern')

spy_min = data_massive.fetch_minute_bars('SPY', APR9)
session = spy_min.between_time('09:30', '16:00')
print(f'{len(session)} minute bars {session.index[0]:%H:%M} → {session.index[-1]:%H:%M} ET')

def pct_move(t0, t1):
    a = session['close'].asof(t0)
    b = session['close'].asof(min(t1, session.index[-1]))
    return b / a - 1.0

apr9_stats = {
    'spy_change_post_to_pause': float(pct_move(POST_TIME, PAUSE_TIME)),
    'spy_change_hour_after_pause': float(pct_move(PAUSE_TIME, PAUSE_TIME + timedelta(hours=1))),
    'spy_change_full_day': float(session['close'].iloc[-1] / session['open'].iloc[0] - 1.0),
    'minutes_post_to_pause': int((PAUSE_TIME - POST_TIME).total_seconds() // 60),
}
print(f"Post → pause: {apr9_stats['minutes_post_to_pause']} minutes")
print(f"SPY 9:37→13:18: {apr9_stats['spy_change_post_to_pause']:+.2%}")
print(f"SPY in the hour after the pause: {apr9_stats['spy_change_hour_after_pause']:+.2%}")
print(f"SPY full session: {apr9_stats['spy_change_full_day']:+.2%}")

391 minute bars 09:30 → 16:00 ET
Post → pause: 221 minutes
SPY 9:37→13:18: +1.13%
SPY in the hour after the pause: +7.43%
SPY full session: +10.31%


In [6]:
# ── Visual 2: SPY intraday, Apr 9 2025, with both moments marked ──────────────
def animate_apr9(session, output_path, audio_wav=None,
                 draw_s=14.0, hold_s=4.0):
    fps = brand.VIDEO_FPS
    n_draw = int(draw_s * fps)
    n_total = n_draw + int(hold_s * fps)

    times = mdates.date2num(session.index.tz_localize(None).to_pydatetime())
    px = session['close'].values

    fig, ax = animation.new_axes('SPY - APRIL 9, 2025 (1-MINUTE BARS)', fontsize=20)
    ax.set_xlim(times[0], times[-1])
    pad = (px.max() - px.min()) * 0.15
    ax.set_ylim(px.min() - pad, px.max() + pad * 1.8)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
    ax.set_xlabel('Eastern Time', color=OFF_WHITE, fontsize=13, fontfamily=VCR)
    ax.set_ylabel('SPY ($)', color=OFF_WHITE, fontsize=13, fontfamily=VCR)
    ax.tick_params(labelsize=11)

    line, = ax.plot([], [], color=GREEN, linewidth=2.0, alpha=0.95, zorder=2)

    events = [
        (POST_TIME,  '9:37 AM - "THIS IS A GREAT\nTIME TO BUY!!!"', GREEN),
        (PAUSE_TIME, '1:18 PM - 90-DAY\nTARIFF PAUSE', RED),
    ]
    marks = []
    for when, label, color in events:
        x = mdates.date2num(when.tz_localize(None).to_pydatetime())
        y = float(session['close'].asof(when))
        vline = ax.axvline(x, color=color, linewidth=1.5, linestyle='--',
                           alpha=0.0, zorder=3)
        dot, = ax.plot([x], [y], 'o', color=color, markersize=9, alpha=0.0, zorder=4)
        txt = ax.annotate(label, xy=(x, y), xytext=(10, 40),
                          textcoords='offset points', color=color, fontsize=13,
                          fontfamily=VCR, alpha=0.0, zorder=5, linespacing=1.4)
        marks.append((x, vline, dot, txt))

    # Shaded hour after the pause + move label, revealed at the end of the draw
    x0 = mdates.date2num(PAUSE_TIME.tz_localize(None).to_pydatetime())
    x1 = mdates.date2num((PAUSE_TIME + timedelta(hours=1)).tz_localize(None).to_pydatetime())
    span = ax.axvspan(x0, x1, color=GREEN, alpha=0.0, zorder=1)
    move_txt = ax.text(
        (x0 + x1) / 2, px.max() + pad * 0.9,
        f"+{apr9_stats['spy_change_hour_after_pause']:.1%} IN 1 HOUR",
        color=GREEN, fontsize=15, ha='center', fontfamily=VCR, alpha=0.0, zorder=5)

    def update(frame):
        f = min(frame, n_draw - 1)
        i = int(f / (n_draw - 1) * (len(px) - 1))
        line.set_data(times[:i + 1], px[:i + 1])
        artists = [line]
        x_now = times[i]
        for x, vline, dot, txt in marks:
            a = animation.smooth_step((x_now - x) / (times[-1] - times[0]) * 30) \
                if x_now >= x else 0.0
            vline.set_alpha(0.8 * a)
            dot.set_alpha(a)
            txt.set_alpha(a)
            artists += [vline, dot, txt]
        a_end = animation.smooth_step((frame - n_draw) / (0.6 * fps))
        span.set_alpha(0.10 * a_end)
        move_txt.set_alpha(a_end)
        artists += [span, move_txt]
        return artists

    ani = mpl_animation.FuncAnimation(fig, update, frames=n_total,
                                      interval=1000 / fps, blit=True)
    print(f'Rendering {output_path.name}  ({n_total} frames)...')
    animation.save_animation(ani, fig, output_path, audio_wav)


DRAW_S, HOLD_S = 14.0, 4.0
# The two marker reveals land where the timestamps sit inside the draw window
session_span = (session.index[-1] - session.index[0]).total_seconds()
t_post = (POST_TIME - session.index[0]).total_seconds() / session_span * DRAW_S
t_pause = (PAUSE_TIME - session.index[0]).total_seconds() / session_span * DRAW_S
wav = render_track(
    [Cue('keystroke', t_post, gain_db=-4), Cue('data_blip', t_post + 0.15, gain_db=-6),
     Cue('keystroke', t_pause, gain_db=-4), Cue('line_swell', t_pause + 0.2, gain_db=-4),
     Cue('resolve_tone', DRAW_S + 0.8, gain_db=-6)],
    duration_s=DRAW_S + HOLD_S,
    out_path=AUDIO_DIR / 'd3_apr9.wav',
)
animate_apr9(session, OUT_DIR / 'spy_intraday_apr9.mp4', audio_wav=wav,
             draw_s=DRAW_S, hold_s=HOLD_S)
session.to_csv(OUT_DIR / 'spy_minute_apr9.csv')

findfont: Failed to find font weight bold, now using 400.


Rendering spy_intraday_apr9.mp4  (1080 frames)...


Saved spy_intraday_apr9.mp4  (0.7 MB)


In [7]:
# ── Headline numbers ──────────────────────────────────────────────────────────
summary = {
    'n_disclosed_trades_api': int(len(trump_raw)),
    'api_note': 'Quiver trumpstocktrades endpoint returns the most recent 200 records; '
                'the full OGE 278-T filing count must be verified against the primary document.',
    'n_clean_buys': int(len(trump_buys)),
    'engine_v1': {
        'avg_trade_return': float(tv1['TradeReturn'].mean()),
        'avg_excess_return': float(tv1['ExcessReturn'].mean()),
        'win_rate': float((tv1['TradeReturn'] > 0).mean()),
        'pct_open_holds': float(tv1['Open'].mean()),
    },
    'engine_v2': {
        'avg_trade_return': float(tv2['TradeReturn'].mean()),
        'avg_excess_return': float(tv2['ExcessReturn'].mean()),
        'win_rate': float((tv2['TradeReturn'] > 0).mean()),
        'pct_open_holds': float(tv2['Open'].mean()),
    },
    'april_9_2025': apr9_stats,
    'hold_days': HOLD_DAYS,
}
with open(OUT_DIR / 'summary_stats.json', 'w') as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))

{
  "n_disclosed_trades_api": 112,
  "api_note": "Quiver trumpstocktrades endpoint returns the most recent 200 records; the full OGE 278-T filing count must be verified against the primary document.",
  "n_clean_buys": 65,
  "engine_v1": {
    "avg_trade_return": 0.0017559042011763503,
    "avg_excess_return": -0.020567582464005897,
    "win_rate": 0.4153846153846154,
    "pct_open_holds": 0.0
  },
  "engine_v2": {
    "avg_trade_return": 0.028468814638884433,
    "avg_excess_return": 0.032999870414978136,
    "win_rate": 0.5384615384615384,
    "pct_open_holds": 1.0
  },
  "april_9_2025": {
    "spy_change_post_to_pause": 0.011308006635109447,
    "spy_change_hour_after_pause": 0.07431040345648388,
    "spy_change_full_day": 0.10307230869001294,
    "minutes_post_to_pause": 221
  },
  "hold_days": 60
}
